# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import collections

# Import DataFrame

In [2]:
sampled_df_filepath = "sampled_dataframe.pkl"

sampled_df = pd.read_pickle(sampled_df_filepath).reset_index(drop=True)

### Getting Unique Vocabulary

In [10]:
vocab = set()

for sublist in sampled_df['lemmas']:
    vocab.update(sublist)

vocab_size = len(vocab)

print(f"Vocab size is {vocab_size} lemmas.")

Vocab size is 159862 lemmas.


### Generating BoW Representations

The following cell determines the minimum vocabulary size needed to cover 80% of all token occurrences in the corpus by calculating cumulative relative word frequencies.

In [9]:
token_frequencies = collections.Counter()

for sentence in sampled_df["lemmas"]:
    token_frequencies.update(sentence)

sorted_counts = np.sort(np.fromiter(token_frequencies.values(), dtype=np.uint32), descending=True)
cumulative_sorted_counts = np.cumsum(sorted_counts)
normalized_cumulative_sorted_counts = cumulative_sorted_counts / cumulative_sorted_counts[-1]

FREQUENCY_THRESHOLD = 0.8
threshold = np.searchsorted(normalized_cumulative_sorted_counts, FREQUENCY_THRESHOLD) + 1

print(f"We will use the most frequent {threshold} lemmas as our vocab and insert an additional unknown character (<UNK>) for everything else.")

We will use the most frequent 1126 lemmas as our vocab and insert an additional unknown character (<UNK>) for everything else.


In [10]:
top_lemmas_with_count = token_frequencies.most_common(threshold)
vocab = [lemma for lemma, _ in top_lemmas_with_count]

vocab_size = len(vocab) + 1  # 1 for <UNK> (vocab_size - 1)
print(f"Vocab size is {vocab_size} lemmas.")

# 1. Build the mapping dictionary
word_to_id = {word: idx for idx, word in enumerate(vocab)}
word_to_id['<UNK>'] = vocab_size - 1

# 2. Build BoW array
bow_representations = np.zeros((sampled_df.shape[0], vocab_size), dtype=np.uint16)
unk_id = word_to_id['<UNK>']

for i, sentence in enumerate(sampled_df['lemmas']):
    for lemma in sentence:
        bow_representations[i, word_to_id.get(lemma, unk_id)] += 1

Vocab size is 1127 lemmas.


In [ ]:
class BoWEncoder():
    def __init__(self, freq_threshold=0.8):
        self.freq_threshold = freq_threshold
        self.word_to_id = {}
        self.vocab = []
        self.vocab_size = 0
        self.unk_id = None

    def _calculate_threshold(self, token_frequencies):
        counts = np.array([count for _, count in token_frequencies.most_common()], dtype=np.uint64)
        cumulative_counts = np.cumsum(counts)
        normalized_cumulative_counts = cumulative_counts / cumulative_counts[-1]

        return int(np.searchsorted(normalized_cumulative_counts, self.freq_threshold) + 1)

    def fit(self, sentences):
        token_frequencies = collections.Counter()
        for sentence in sentences:
            token_frequencies.update(sentence)

        threshold = self._calculate_threshold(token_frequencies)
        top_lemmas_with_count = token_frequencies.most_common(threshold)

        self.vocab = [lemma for lemma, _ in top_lemmas_with_count]
        self.unk_id = len(self.vocab)
        self.vocab_size = len(self.vocab) + 1 # 1 for <UNK> (vocab_size - 1)

        self.word_to_id = {word: idx for idx, word in enumerate(self.vocab)}
        self.word_to_id['<UNK>'] = self.unk_id

        return self

    def transform(self, sentences):
        if not self.word_to_id:
            raise RuntimeError("BoWEncoder must be fitted before calling transform().")

        num_samples = len(sentences)
        bow_representations = np.zeros((num_samples, self.vocab_size), dtype=np.uint16)

        word_mapping = self.word_to_id
        unk_idx = self.unk_id

        for i, sentence in enumerate(sentences):
            for lemma in sentence:
                bow_representations[i, word_mapping.get(lemma, unk_idx)] += 1

        return bow_representations

    def fit_transform(self, sentences):
        return self.fit(sentences).transform(sentences)

# Usage:
# encoder = BoWEncoder(freq_threshold=0.8)
# X_train_bow = encoder.fit_transform(train_df['lemmas'])
# X_test_bow = encoder.transform(test_df['lemmas'])

# 1D Convolutions

In [5]:
class SequenceEncoder():
    def __init__(self, freq_threshold=0.8, max_length=50):
        self.freq_threshold = freq_threshold
        self.word_to_id = {}
        self.vocab = []
        self.vocab_size = 0
        self.pad_id = 0
        self.unk_id = None
        self.max_length = max_length

    def _calculate_threshold(self, token_frequencies):
        counts = np.array([count for _, count in token_frequencies.most_common()], dtype=np.uint64)
        cumulative_counts = np.cumsum(counts)
        normalized_cumulative_counts = cumulative_counts / cumulative_counts[-1]

        return int(np.searchsorted(normalized_cumulative_counts, self.freq_threshold) + 1)

    def fit(self, sentences):
        token_frequencies = collections.Counter()
        for sentence in sentences:
            token_frequencies.update(sentence)

        threshold = self._calculate_threshold(token_frequencies)
        top_lemmas_with_count = token_frequencies.most_common(threshold)

        self.vocab = [lemma for lemma, _ in top_lemmas_with_count]
        self.vocab_size = len(self.vocab) + 2 # 1 for <UNK> and 1 for <PAD>
        self.unk_id = self.vocab_size - 1

        self.word_to_id = {word: idx for idx, word in enumerate(self.vocab, start=1)}
        self.word_to_id['<PAD>'] = self.pad_id
        self.word_to_id['<UNK>'] = self.unk_id

        return self

    def transform(self, sentences):
        if not self.word_to_id:
            raise RuntimeError("SequenceEncoder must be fitted before calling transform().")

        num_samples = len(sentences)
        int_representations = np.zeros((num_samples, self.max_length), dtype=np.uint32)

        word_mapping = self.word_to_id
        unk_idx = self.unk_id
        max_len = self.max_length

        for i, sentence in enumerate(sentences):
            truncated = sentence[:max_len]
            for j, lemma in enumerate(truncated):
                int_representations[i, j] = word_mapping.get(lemma, unk_idx)

        return int_representations

    def fit_transform(self, sentences):
        return self.fit(sentences).transform(sentences)

# Usage:
# encoder = SequenceEncoder(freq_threshold=0.8, max_length=50)
# X_train_bow = encoder.fit_transform(train_df['lemmas'])
# X_test_bow = encoder.transform(test_df['lemmas'])

In [6]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()

target_column = 'Party_orientation'
sentence_column = 'lemmas'
y = le.fit_transform(sampled_df[target_column])
X_lemmas = sampled_df[sentence_column]

X_train_lemmas, X_test_lemmas, y_train, y_test = train_test_split(X_lemmas, y, test_size=0.1, random_state=42, stratify=y)

# 3. Fit Encoder ONLY on Train, Transform Both
encoder = SequenceEncoder(freq_threshold=0.8, max_length=50)
X_train_int = encoder.fit_transform(X_train_lemmas)
X_test_int = encoder.transform(X_test_lemmas)

print(f"Vocab size (including PAD & UNK): {encoder.vocab_size}")
print(f"X_train shape: {X_train_int.shape}")
print(f"X_test shape:  {X_test_int.shape}")

Vocab size (including PAD & UNK): 1129
X_train shape: (458964, 50)
X_test shape:  (50996, 50)


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_tensor_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_tensor_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 128
train_loader = DataLoader(train_tensor_dataset, batch_size, shuffle=True)
test_loader = DataLoader(test_tensor_dataset, batch_size, shuffle=True)

Defining Model Architecture

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvNet(nn.Module):
    def __init__(self, vocab_size, num_classes=9, output_channels=64, kernel_size=3):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv1d(vocab_size, output_channels, kernel_size)
        self.pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.fc1 = nn.Linear(output_channels, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1) # (batch, max_length, vocab_size) -> (batch, vocab_size, max_length)

        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)

        x = x.squeeze(-1) # (batch, 64, 1) -> (batch, 64)
        x = self.fc1(x)

        return x

In [26]:
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_tensor_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_tensor_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 128
train_loader = DataLoader(train_tensor_dataset, batch_size, shuffle=True)
test_loader = DataLoader(test_tensor_dataset, batch_size, shuffle=True)

In [28]:
import torch.optim as optim

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = ConvNet(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Test for one batch

In [33]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

num_epochs = 5

for epoch in range(num_epochs):
    # -----------------------------
    # 1. TRAINING PHASE
    # -----------------------------
    model.train()
    
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    # Wrap the train_loader with tqdm
    train_pbar = tqdm(
        train_loader, 
        desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", 
        leave=False
    )

    for batch_x, batch_y in train_pbar:
        # On-the-fly one-hot encoding & device placement
        batch_x = F.one_hot(batch_x, num_classes=vocab_size).float().to(device)
        batch_y = batch_y.to(device)

        # Optimization step
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        # Running metrics tracking
        batch_size = batch_y.size(0)
        running_train_loss += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)
        correct_train += (predictions == batch_y).sum().item()
        total_train += batch_size

        # Update live progress bar display
        current_avg_loss = running_train_loss / total_train
        current_acc = (correct_train / total_train) * 100
        train_pbar.set_postfix({
            "batch_loss": f"{loss.item():.4f}",
            "avg_loss": f"{current_avg_loss:.4f}",
            "acc": f"{current_acc:.2f}%"
        })

    epoch_train_loss = running_train_loss / total_train
    epoch_train_acc = correct_train / total_train

    # -----------------------------
    # 2. VALIDATION PHASE
    # -----------------------------
    model.eval()
    
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    val_pbar = tqdm(
        test_loader, 
        desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", 
        leave=False
    )

    with torch.no_grad():
        for batch_x, batch_y in val_pbar:
            batch_x = F.one_hot(batch_x, num_classes=vocab_size).float().to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            batch_size = batch_y.size(0)
            running_val_loss += loss.item() * batch_size
            predictions = outputs.argmax(dim=1)
            correct_val += (predictions == batch_y).sum().item()
            total_val += batch_size

            val_pbar.set_postfix({
                "val_loss": f"{loss.item():.4f}",
                "val_acc": f"{(correct_val / total_val) * 100:.2f}%"
            })

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = correct_val / total_val

    # -----------------------------
    # 3. EPOCH SUMMARY
    # -----------------------------
    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc * 100:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc * 100:.2f}%"
    )

/Users/mikhail/Documents/Masters/NLP 1/project/nlp-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch [1/5] | Train Loss: 1.4563 - Train Acc: 44.88% | Val Loss: 1.4997 - Val Acc: 42.75%


Epoch [2/5] | Train Loss: 1.4141 - Train Acc: 46.78% | Val Loss: 1.4994 - Val Acc: 42.98%


Epoch [3/5] | Train Loss: 1.3812 - Train Acc: 48.22% | Val Loss: 1.5073 - Val Acc: 42.65%


Epoch [4/5] | Train Loss: 1.3540 - Train Acc: 49.43% | Val Loss: 1.5203 - Val Acc: 42.24%


Epoch [5/5] | Train Loss: 1.3320 - Train Acc: 50.35% | Val Loss: 1.5300 - Val Acc: 41.91%


Lets make the model deeper

In [34]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvNet(nn.Module):
    def __init__(self, vocab_size, num_classes=9, output_channels=64, kernel_size=3):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv1d(vocab_size, output_channels, kernel_size, padding='same')
        self.conv2 = nn.Conv1d(output_channels, output_channels, kernel_size, padding='same')
        self.conv3 = nn.Conv1d(output_channels, output_channels, kernel_size, padding='same')
        self.pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(output_channels, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1) # (batch, max_length, vocab_size) -> (batch, vocab_size, max_length)

        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)

        x = self.conv3(x)
        x = F.relu(x)

        x = self.pool(x)


        x = x.squeeze(-1) # (batch, 64, 1) -> (batch, 64)

        x = self.dropout(x)

        x = self.fc1(x)

        return x

In [35]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

num_epochs = 5

for epoch in range(num_epochs):
    # -----------------------------
    # 1. TRAINING PHASE
    # -----------------------------
    model.train()
    
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    # Wrap the train_loader with tqdm
    train_pbar = tqdm(
        train_loader, 
        desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", 
        leave=False
    )

    for batch_x, batch_y in train_pbar:
        # On-the-fly one-hot encoding & device placement
        batch_x = F.one_hot(batch_x, num_classes=vocab_size).float().to(device)
        batch_y = batch_y.to(device)

        # Optimization step
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        # Running metrics tracking
        batch_size = batch_y.size(0)
        running_train_loss += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)
        correct_train += (predictions == batch_y).sum().item()
        total_train += batch_size

        # Update live progress bar display
        current_avg_loss = running_train_loss / total_train
        current_acc = (correct_train / total_train) * 100
        train_pbar.set_postfix({
            "batch_loss": f"{loss.item():.4f}",
            "avg_loss": f"{current_avg_loss:.4f}",
            "acc": f"{current_acc:.2f}%"
        })

    epoch_train_loss = running_train_loss / total_train
    epoch_train_acc = correct_train / total_train

    # -----------------------------
    # 2. VALIDATION PHASE
    # -----------------------------
    model.eval()
    
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    val_pbar = tqdm(
        test_loader, 
        desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", 
        leave=False
    )

    with torch.no_grad():
        for batch_x, batch_y in val_pbar:
            batch_x = F.one_hot(batch_x, num_classes=vocab_size).float().to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            batch_size = batch_y.size(0)
            running_val_loss += loss.item() * batch_size
            predictions = outputs.argmax(dim=1)
            correct_val += (predictions == batch_y).sum().item()
            total_val += batch_size

            val_pbar.set_postfix({
                "val_loss": f"{loss.item():.4f}",
                "val_acc": f"{(correct_val / total_val) * 100:.2f}%"
            })

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = correct_val / total_val

    # -----------------------------
    # 3. EPOCH SUMMARY
    # -----------------------------
    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc * 100:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc * 100:.2f}%"
    )

Epoch [1/5] | Train Loss: 1.3134 - Train Acc: 51.21% | Val Loss: 1.5430 - Val Acc: 41.54%


Epoch [2/5] | Train Loss: 1.2978 - Train Acc: 51.82% | Val Loss: 1.5575 - Val Acc: 41.45%


Epoch [3/5] | Train Loss: 1.2843 - Train Acc: 52.41% | Val Loss: 1.5729 - Val Acc: 40.81%


Epoch [4/5] | Train Loss: 1.2732 - Train Acc: 52.92% | Val Loss: 1.5833 - Val Acc: 41.08%


Epoch [5/5] | Train Loss: 1.2633 - Train Acc: 53.34% | Val Loss: 1.5948 - Val Acc: 40.71%


# Lets try to fix sparsity by training an embedding layer instead of using token indicators

In [8]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train_int, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_int, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_tensor_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_tensor_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 128
train_loader = DataLoader(train_tensor_dataset, batch_size, shuffle=True)
test_loader = DataLoader(test_tensor_dataset, batch_size, shuffle=False)

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EmbeddingConvNet(nn.Module):
    def __init__(self, vocab_size, num_classes=9, embedding_dim=256, output_channels=64, kernel_size=3, padding_idx=0):
        super(EmbeddingConvNet, self).__init__()
        self.emb = nn.Embedding(vocab_size, embedding_dim, padding_idx)
        self.conv1 = nn.Conv1d(embedding_dim, output_channels, kernel_size, padding='same')
        self.conv2 = nn.Conv1d(output_channels, output_channels, kernel_size, padding='same')
        self.conv3 = nn.Conv1d(output_channels, output_channels, kernel_size, padding='same')
        self.pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(output_channels, num_classes)

    def forward(self, x):
        x = self.emb(x) # (batch, max_length) -> (batch, max_length, vocab_size)
        x = x.permute(0, 2, 1) # (batch, max_length, vocab_size) -> (batch, vocab_size, max_length)

        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)

        x = self.conv3(x)
        x = F.relu(x)

        x = self.pool(x)


        x = x.squeeze(-1) # (batch, 64, 1) -> (batch, 64)

        x = self.dropout(x)

        x = self.fc1(x)

        return x

In [18]:
import torch.optim as optim

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = EmbeddingConvNet(vocab_size=encoder.vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [19]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

num_epochs = 5

for epoch in range(num_epochs):
    # -----------------------------
    # 1. TRAINING PHASE
    # -----------------------------
    model.train()
    
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    # Wrap the train_loader with tqdm
    train_pbar = tqdm(
        train_loader, 
        desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", 
        leave=False
    )

    for batch_x, batch_y in train_pbar:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Optimization step
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        # Running metrics tracking
        batch_size = batch_y.size(0)
        running_train_loss += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)
        correct_train += (predictions == batch_y).sum().item()
        total_train += batch_size

        # Update live progress bar display
        current_avg_loss = running_train_loss / total_train
        current_acc = (correct_train / total_train) * 100
        train_pbar.set_postfix({
            "batch_loss": f"{loss.item():.4f}",
            "avg_loss": f"{current_avg_loss:.4f}",
            "acc": f"{current_acc:.2f}%"
        })

    epoch_train_loss = running_train_loss / total_train
    epoch_train_acc = correct_train / total_train

    # -----------------------------
    # 2. VALIDATION PHASE
    # -----------------------------
    model.eval()
    
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    val_pbar = tqdm(
        test_loader, 
        desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", 
        leave=False
    )

    with torch.no_grad():
        for batch_x, batch_y in val_pbar:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            batch_size = batch_y.size(0)
            running_val_loss += loss.item() * batch_size
            predictions = outputs.argmax(dim=1)
            correct_val += (predictions == batch_y).sum().item()
            total_val += batch_size

            val_pbar.set_postfix({
                "val_loss": f"{loss.item():.4f}",
                "val_acc": f"{(correct_val / total_val) * 100:.2f}%"
            })

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = correct_val / total_val

    # -----------------------------
    # 3. EPOCH SUMMARY
    # -----------------------------
    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc * 100:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc * 100:.2f}%"
    )

Epoch [1/5] | Train Loss: 1.6213 - Train Acc: 36.23% | Val Loss: 1.5610 - Val Acc: 39.25%


Epoch [2/5] | Train Loss: 1.5601 - Train Acc: 39.68% | Val Loss: 1.5345 - Val Acc: 41.06%


Epoch [3/5] | Train Loss: 1.5347 - Train Acc: 40.97% | Val Loss: 1.5202 - Val Acc: 41.78%


Epoch [4/5] | Train Loss: 1.5176 - Train Acc: 41.68% | Val Loss: 1.5180 - Val Acc: 41.96%


Epoch [5/5] | Train Loss: 1.5046 - Train Acc: 42.36% | Val Loss: 1.5073 - Val Acc: 42.12%


Hyperparameter tuning with optuna

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FlexibleConvNet(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int = 9,
        embedding_dim: int = 64,
        output_channels: int = 64,
        kernel_size: int = 3,
        num_layers: int = 2,
        dropout_rate: float = 0.3,
        padding_idx: int = 0
    ):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)

        # Layer 1: embedding_dim -> output_channels
        layers = [
            nn.Conv1d(embedding_dim, output_channels, kernel_size, padding='same'),
            nn.ReLU()
        ]

        # Layers 2..N: output_channels -> output_channels
        for _ in range(num_layers - 1):
            layers.extend([
                nn.Conv1d(output_channels, output_channels, kernel_size, padding='same'),
                nn.ReLU()
            ])

        self.conv_stack = nn.Sequential(*layers)
        self.pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(output_channels, num_classes)

    def forward(self, x):
        x = self.emb(x)             # (batch, seq_len) -> (batch, seq_len, emb_dim)
        x = x.permute(0, 2, 1)      # (batch, seq_len, emb_dim) -> (batch, emb_dim, seq_len)
        x = self.conv_stack(x)      # (batch, emb_dim, seq_len) -> (batch, out_channels, seq_len)
        x = self.pool(x).squeeze(-1)# (batch, out_channels, 1) -> (batch, out_channels)
        x = self.dropout(x)
        return self.fc1(x)          # (batch, num_classes)

In [26]:
import optuna
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from tqdm.auto import tqdm


def objective(trial):
    # 1. Hyperparameters
    embedding_dim = trial.suggest_categorical("embedding_dim", [32, 64, 128])
    output_channels = trial.suggest_categorical("output_channels", [64, 128, 256])
    kernel_size = trial.suggest_categorical("kernel_size", [3, 5])
    num_layers = trial.suggest_int("num_layers", 2, 3)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 2e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    # 2. Model & Optimizer
    model = FlexibleConvNet(
        vocab_size=encoder.vocab_size,
        num_classes=9,
        embedding_dim=embedding_dim,
        output_channels=output_channels,
        kernel_size=kernel_size,
        num_layers=num_layers,
        dropout_rate=dropout_rate,
        padding_idx=0
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    epochs_per_trial = 4

    for epoch in range(epochs_per_trial):
        # -----------------------------
        # TRAINING PHASE
        # -----------------------------
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        # Dedicated bar for training
        pbar = tqdm(
            train_loader,
            desc=f"Trial {trial.number:02d} | Ep {epoch + 1}/{epochs_per_trial}",
            leave=False
        )

        for batch_x, batch_y in pbar:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            batch_size = batch_y.size(0)
            running_loss += loss.item() * batch_size
            preds = outputs.argmax(dim=1)
            correct_train += (preds == batch_y).sum().item()
            total_train += batch_size

            # Update live stats smoothly
            pbar.set_postfix({
                "loss": f"{running_loss / total_train:.4f}",
                "acc": f"{(correct_train / total_train) * 100:.1f}%"
            })

        pbar.close()  # Cleanly close before evaluation

        epoch_train_loss = running_loss / total_train
        epoch_train_acc = (correct_train / total_train) * 100

        # -----------------------------
        # VALIDATION PHASE (Silent to prevent UI jitter)
        # -----------------------------
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_preds, val_targets = [], []

        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)

                batch_size = batch_y.size(0)
                val_loss += loss.item() * batch_size
                preds = outputs.argmax(dim=1)
                val_correct += (preds == batch_y).sum().item()
                val_total += batch_size

                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())

        epoch_val_loss = val_loss / val_total
        epoch_val_acc = (val_correct / val_total) * 100
        val_macro_f1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)

        # USE tqdm.write INSTEAD OF print()
        tqdm.write(
            f"Trial {trial.number:02d} | Epoch {epoch + 1}/{epochs_per_trial} | "
            f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.1f}% | "
            f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.1f}% | "
            f"Val F1: {val_macro_f1:.4f}"
        )

        # -----------------------------
        # PRUNING CHECK
        # -----------------------------
        trial.report(val_macro_f1, epoch)
        if trial.should_prune():
            tqdm.write(f"--> Trial {trial.number:02d} PRUNED at epoch {epoch + 1}")
            raise optuna.exceptions.TrialPruned()

    return val_macro_f1

In [27]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)

# Set show_progress_bar=False so only your inner pbar updates the screen
study.optimize(objective, n_trials=20, show_progress_bar=False)

Trial 00 | Epoch 1/4 | Train Loss: 1.7193 Acc: 31.1% | Val Loss: 1.6652 Acc: 34.3% | Val F1: 0.1126


Trial 00 | Epoch 2/4 | Train Loss: 1.6641 Acc: 34.0% | Val Loss: 1.6381 Acc: 35.2% | Val F1: 0.1110


Trial 00 | Epoch 3/4 | Train Loss: 1.6426 Acc: 35.0% | Val Loss: 1.6233 Acc: 36.0% | Val F1: 0.1260


Trial 00 | Epoch 4/4 | Train Loss: 1.6286 Acc: 35.6% | Val Loss: 1.6125 Acc: 36.5% | Val F1: 0.1273


Trial 01 | Epoch 1/4 | Train Loss: 1.6492 Acc: 34.7% | Val Loss: 1.5914 Acc: 37.8% | Val F1: 0.1502


Trial 01 | Epoch 2/4 | Train Loss: 1.5748 Acc: 38.7% | Val Loss: 1.5528 Acc: 39.8% | Val F1: 0.1704


... (61 outputs left out)

Trial 19 | Epoch 1/4 | Train Loss: 1.6160 Acc: 36.7% | Val Loss: 1.5579 Acc: 39.8% | Val F1: 0.1751


Trial 19 | Epoch 2/4 | Train Loss: 1.5387 Acc: 40.6% | Val Loss: 1.5230 Acc: 41.3% | Val F1: 0.1952


Trial 19 | Epoch 3/4 | Train Loss: 1.5038 Acc: 42.2% | Val Loss: 1.5117 Acc: 42.0% | Val F1: 0.2012


Trial 19 | Epoch 4/4 | Train Loss: 1.4793 Acc: 43.4% | Val Loss: 1.4965 Acc: 42.8% | Val F1: 0.2185
